# Session context check: when does history help?

Exploratory follow-up to the interim signal checks (4 October 2026). **Question:** does the ranking value of a shopper's history depend on the search context, such as how long ago their last action was, whether a past query shares a term with the current one, query length, category diversity of the list, and history length?

**Method.** Reuse the per-search NDCG@10 of single-signal rankings from the two validation notebooks: same 5,000-record sample (seed 4222), the 4,941 searches with an engaged candidate, linear gains 0/1/2/3 and expected random tie-breaking. *Gain* = signal NDCG@10 minus expected random NDCG@10. *Time since last action* = the final element of `history_time_list`, in seconds.

Run `query_signal_validation.ipynb` and `history_match_validation.ipynb` first; this notebook reads their saved tables plus the raw behaviour file. Results are untrained heuristics; see the caveats at the end.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_ROOT = next((p.resolve() for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/interim_eda.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open/clone the repository checkout first.")
if IN_COLAB:
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/BT4222 Group Project/data")
else:
    DATA_DIR = REPO_ROOT / "data"
TABLES = REPO_ROOT / "outputs/interim/tables"
SEED = 4222
pd.options.display.float_format = "{:,.3f}".format

In [2]:
history = pd.read_csv(TABLES / "history_validation/history_match_per_search.csv")
query = pd.read_csv(TABLES / "query_validation/query_diagnostics_per_search.csv")
for summary in ["history_validation/history_match_summary.json", "query_validation/query_diagnostics_summary.json"]:
    meta = json.loads((TABLES / summary).read_text())
    assert meta["seed"] == SEED and meta["sample_records"] == 5000
assert history.session_id.equals(query.session_id)

searches = history.merge(
    query[["session_id", "query_terms", "ndcg_title_jaccard", "history_any_token_overlap"]], on="session_id")
searches = searches[searches.positive_candidates > 0].copy()
print(f"Searches with an engaged candidate: {len(searches):,}")

Searches with an engaged candidate: 4,941


In [3]:
# The final element of history_time_list is the gap from the last action to the current search.
time_lists = pd.read_csv(DATA_DIR / "user_behavior_data.txt", sep="	",
                         usecols=["history_time_list"], dtype=str)["history_time_list"]
gap_lists = [time_lists.iloc[i] for i in searches.session_id]
# Row alignment check: one gap per historical action plus the final gap to the search.
assert all(g.count("_") + 1 == n + 1 for g, n in zip(gap_lists, searches.history_events))
searches["since_last_s"] = [int(g.rsplit("_", 1)[-1]) for g in gap_lists]
searches.since_last_s.describe(percentiles=[.25, .5, .75, .95])

count        4,941.000
mean       511,122.486
std      2,038,510.524
min              1.000
25%             67.000
50%         21,789.000
75%        205,226.000
95%      2,413,022.000
max     31,307,625.000
Name: since_last_s, dtype: float64

In [4]:
signals = {"same product": "all_product_ndcg", "same shop": "all_shop_ndcg", "same brand": "all_brand_ndcg",
           "same L3 path": "all_L3_ndcg", "query-title Jaccard": "ndcg_title_jaccard"}
for name, column in signals.items():
    searches[name] = searches[column] - searches.ndcg_random


def gains_by(groups):
    """Mean NDCG@10 gain over random order per signal, by search segment."""
    grouped = searches.groupby(groups, observed=True)
    table = grouped[list(signals)].mean()
    table.insert(0, "searches", grouped.size())
    return table

## Gains by search context

### Time since the last action

In [5]:
recency = pd.cut(searches.since_last_s, [-1, 600, 6 * 3600, 7 * 86400, np.inf],
                 labels=["< 10 min", "10 min - 6 h", "6 h - 7 d", "> 7 d"]).rename("time since last action")
gains_by(recency)

,searches,same product,same shop,same brand,same L3 path,query-title Jaccard
time since last action,,,,,,
< 10 min,1940,0.181,0.141,0.069,0.021,0.013
10 min - 6 h,527,0.079,0.091,0.039,0.013,0.013
6 h - 7 d,1868,0.046,0.056,0.016,0.010,0.013
> 7 d,606,0.025,0.029,0.012,0.003,0.022


### Overlap between a past query and the current query, alone and crossed with recency

In [6]:
searches["within 10 min"] = searches.since_last_s < 600
display(gains_by(searches.history_any_token_overlap.rename("past query shares a term")))
display(gains_by(["within 10 min", "history_any_token_overlap"]))

,searches,same product,same shop,same brand,same L3 path,query-title Jaccard
past query shares a term,,,,,,
False,2909,0.058,0.058,0.025,0.009,0.013
True,2032,0.160,0.135,0.059,0.021,0.015


searches  same product  same shop  \
within 10 min history_any_token_overlap                                      
False         False                          2137         0.028      0.038   
              True                            864         0.096      0.103   
True          False                           772         0.142      0.113   
              True                           1168         0.208      0.159   

                                         same brand  same L3 path  \
within 10 min history_any_token_overlap                             
False         False                           0.013         0.005   
              True                            0.034         0.017   
True          False                           0.056         0.018   
              True                            0.078         0.023   

                                         query-title Jaccard  
within 10 min history_any_token_overlap                       
False         False                                    0.014  
              True                                     0.016  
True          False                                    0.010  
              True                                     0.014

### Query length, category diversity of the list, history length

In [7]:
display(gains_by(pd.cut(searches.query_terms, [0, 1, 2, 3, np.inf], labels=["1", "2", "3", "4+"]).rename("query terms")))
display(gains_by(searches.L1_attribute_varies.rename("candidates span > 1 L1 category")))
display(gains_by(pd.cut(searches.history_events, [0, 30, 100, 300, np.inf],
                        labels=["<= 30", "31-100", "101-300", "> 300"]).rename("history actions")))

,searches,same product,same shop,same brand,same L3 path,query-title Jaccard
query terms,,,,,,
1,974,0.096,0.078,0.038,0.014,-0.003
2,1612,0.085,0.087,0.041,0.010,0.008
3,1152,0.103,0.097,0.040,0.015,0.017
4+,1203,0.121,0.095,0.036,0.017,0.032


,searches,same product,same shop,same brand,same L3 path,query-title Jaccard
candidates span > 1 L1 category,,,,,,
False,1665,0.095,0.082,0.037,0.007,0.021
True,3276,0.103,0.094,0.040,0.017,0.010


,searches,same product,same shop,same brand,same L3 path,query-title Jaccard
history actions,,,,,,
<= 30,1303,0.086,0.075,0.033,0.015,0.014
31-100,1568,0.103,0.086,0.036,0.011,0.012
101-300,1408,0.097,0.098,0.045,0.013,0.016
> 300,662,0.128,0.109,0.044,0.019,0.012


In [8]:
rng = np.random.default_rng(SEED)
recent = searches.loc[searches["within 10 min"], "same shop"].to_numpy()
older = searches.loc[~searches["within 10 min"], "same shop"].to_numpy()
diffs = [rng.choice(recent, len(recent)).mean() - rng.choice(older, len(older)).mean() for _ in range(2000)]
print(f"Searches within 10 minutes of the last action: {searches['within 10 min'].mean():.1%}")
print(f"Same-shop gain, within 10 min minus later: {recent.mean() - older.mean():.3f} "
      f"(95% bootstrap interval {np.percentile(diffs, 2.5):.3f} to {np.percentile(diffs, 97.5):.3f})")

Searches within 10 minutes of the last action: 39.3%
Same-shop gain, within 10 min minus later: 0.084 (95% bootstrap interval 0.072 to 0.096)


## Reading and caveats

- **Recency is the strongest moderator found so far.** History-based rankings gain far more over random order when the search follows the last action within minutes than when the shopper returns days later, while the query-title gain stays roughly flat. This motivates the *session-aware personalisation* hypothesis in `docs/FINDINGS.md`.
- **Past-query overlap is a second, separate moderator.** It holds within both recency groups. It supports weighting history by query relevance rather than hard-filtering it.
- **Query length and category diversity do not support the classic "personalise ambiguous queries" idea** in this check.
- **Caveats.** Untrained single-signal rankings on a 4,941-search sample; segments are correlated and not causal. Checks still owed: confirm that history strictly precedes the search, exclude exact re-finds of products just interacted with, rerun on a larger sample, and test the interaction inside a trained model.